# GNNocRoute-DRL: Colab Worker (Drive Queue)
## Polls Drive for tasks from server, runs training, saves results
---
**How it works:**
1. Server writes `.task.json` to Drive
2. This notebook checks Drive every 30s
3. Executes training/energy experiments
4. Writes `.result.json` back to Drive
5. Server reads results

**Run this notebook and leave it running!**
Runtime → Run all → Colab will process tasks from the server.

In [ ]:
# === SETUP ===
import os, sys, json, time, random, re, subprocess, tempfile, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from datetime import datetime
from google.colab import drive

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

# Mount Drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'

# Queue paths
QUEUE_DIR = f'{BASE}/GNNocRoute_Research_Backup/hosoncs/2025/colab-queue'
TASKS_DIR = f'{QUEUE_DIR}/tasks'
RESULTS_DIR = f'{QUEUE_DIR}/results'
STATUS_DIR = f'{QUEUE_DIR}/status'
for d in [TASKS_DIR, RESULTS_DIR, STATUS_DIR, f'{BASE}/GNNocRoute_DRL_Results/models']:
    os.makedirs(d, exist_ok=True)

print(f'Queue: {QUEUE_DIR}')
print(f'Device: {DEVICE}')
print('\n✅ Setup complete! Waiting for tasks...')

In [ ]:
# === GNN + DQN MODELS ===
class GNNEncoder(nn.Module):
    def __init__(self, in_dim=4, hidden=64, out=32):
        super().__init__()
        from torch_geometric.nn import GATv2Conv
        self.conv1 = GATv2Conv(in_dim, hidden//4, heads=4)
        self.conv2 = GATv2Conv(hidden, out, heads=1)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(out)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index); x = self.norm1(x); x = F.elu(x)
        x = self.conv2(x, edge_index); x = self.norm2(x)
        return x

class DQN(nn.Module):
    def __init__(self, in_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(), nn.LayerNorm(64),
            nn.Linear(64, 4))
    def forward(self, x): return self.net(x)

print('✅ Models ready')

In [ ]:
# === TRAINING FUNCTION ===
def run_drl_training(params):
    """Run DRL training with given parameters."""
    traffic = params.get('traffic', 'hotspot')
    episodes = params.get('episodes', 200)
    profile = params.get('profile', 'blackscholes')
    inj_rate = params.get('inj_rate', 0.1)
    
    print(f'  Training: {traffic}/{profile} ({episodes} eps)')
    
    gnn = GNNEncoder().to(DEVICE)
    dqn = DQN().to(DEVICE)
    target = DQN().to(DEVICE)
    target.load_state_dict(dqn.state_dict())
    opt = torch.optim.Adam(list(gnn.parameters())+list(dqn.parameters()), lr=5e-4)
    
    # Simple env
    from torch_geometric.utils import grid_graph
    edge_index = grid_graph((4,4)).to(DEVICE)
    
    memory = deque(maxlen=10000)
    rewards = []
    epsilon = 0.5
    
    t0 = time.time()
    for ep in range(episodes):
        s = torch.randn(16, 4).to(DEVICE)
        ep_r = 0
        for step in range(100):
            with torch.no_grad():
                if random.random() < epsilon:
                    a = torch.randint(0, 4, (16,)).to(DEVICE)
                else:
                    a = dqn(gnn(s, edge_index)).argmax(dim=1)
            
            ns = torch.randn(16, 4).to(DEVICE)
            reward = -torch.mean(s[:,0]).item() * 2 + 0.1
            
            for i in range(16):
                memory.append((s[i].cpu().numpy(), a[i].item(), reward/16, ns[i].cpu().numpy(), step>=99))
            
            s, ep_r = ns, ep_r + reward
            
            if len(memory) >= 64:
                batch = random.sample(list(memory), 64)
                ss = torch.FloatTensor(np.array([b[0] for b in batch])).to(DEVICE)
                aa = torch.LongTensor([b[1] for b in batch]).unsqueeze(1).to(DEVICE)
                rr = torch.FloatTensor([[b[2]] for b in batch]).to(DEVICE)
                nss = torch.FloatTensor(np.array([b[3] for b in batch])).to(DEVICE)
                
                with torch.no_grad():
                    tgt = rr + 0.95 * target(gnn(nss, edge_index)).max(1,keepdim=True)[0]
                loss = F.mse_loss(dqn(gnn(ss, edge_index)).gather(1, aa), tgt)
                opt.zero_grad(); loss.backward(); opt.step()
        
        epsilon = max(0.01, epsilon - 0.5/episodes)
        rewards.append(ep_r)
        if ep % 10 == 0: target.load_state_dict(dqn.state_dict())
        if (ep+1) % 100 == 0:
            print(f'    Ep {ep+1}/{episodes} | R={np.mean(rewards[-50:]):.2f} | eps={epsilon:.3f}')
    
    elapsed = time.time() - t0
    avg_r = float(np.mean(rewards[-50:]))
    
    # Save to Drive
    model_path = f'{BASE}/GNNocRoute_DRL_Results/models/drl_{traffic}_{profile}.pt'
    torch.save({'gnn':gnn.state_dict(),'dqn':dqn.state_dict(),'rewards':rewards}, model_path)
    
    return {
        'status': 'completed',
        'avg_reward': avg_r,
        'time_min': elapsed/60,
        'episodes': episodes,
        'model_path': model_path
    }

print('✅ Training function ready')

In [ ]:
# === BOOKSIM2 ENERGY FUNCTION ===
def run_booksim_energy(params):
    """Run BookSim2 energy experiments."""
    print('  Installing BookSim2...')
    if not os.path.exists('/content/booksim2'):
        !git clone https://github.com/booksim/booksim2.git /content/booksim2 2>&1 | tail -1
        %cd /content/booksim2/src
        !make 2>&1 | tail -3
    
    BOOKSIM = '/content/booksim2/src/booksim'
    if not os.path.exists(BOOKSIM):
        return {'status': 'failed', 'error': 'BookSim2 build failed'}
    
    results = []
    topos = params.get('topologies', ['mesh44','mesh88'])
    algos = params.get('algorithms', ['dor','adaptive_xy_yx','min_adapt'])
    traffics = params.get('traffics', ['uniform','transpose','hotspot'])
    injs = params.get('inj_rates', [0.01, 0.02, 0.05, 0.1])
    
    for topo in topos:
        k = 4 if '44' in topo else 8
        for algo in algos:
            for traffic in traffics:
                for inj in injs:
                    cfg = tempfile.NamedTemporaryFile(mode='w', suffix='.cfg', delete=False)
                    cfg.write(f'''topology = mesh; k = {k}; n = 2;
routing_function = {algo};
num_vcs = 4; vc_buf_size = 8;
traffic = {traffic}; injection_rate = {inj};
sim_type = latency; warmup_periods = 3;
sample_period = 1000; sim_count = 1;
power_output_file = /tmp/power.txt;
sim_power = 1;
tech_file = /content/booksim2/src/power/techfile.txt;
''')
                    cfg.close()
                    try:
                        r = subprocess.run([BOOKSIM, cfg.name], capture_output=True, text=True, timeout=30)
                        lat = re.search(r'Packet latency average\s*=\s*([0-9.]+)', r.stdout)
                        en = re.search(r'Total energy\s*=\s*([0-9.]+)', r.stdout)
                        lat_v = float(lat.group(1)) if lat else None
                        en_v = float(en.group(1)) if en else None
                        results.append({'topology':topo,'algorithm':algo,'traffic':traffic,
                                       'inj_rate':inj,'latency':lat_v,'energy':en_v,'status':'OK'})
                        s = f'{topo} {algo} {traffic} @{inj}'
                        print(f'    {s:45s} lat={lat_v} en={en_v}')
                    except Exception as e:
                        results.append({'topology':topo,'algorithm':algo,'traffic':traffic,
                                       'inj_rate':inj,'latency':None,'energy':None,'status':str(e)})
                    os.unlink(cfg.name)
    
    return {'status': 'completed', 'results': results}

print('✅ BookSim2 function ready')

In [ ]:
# === MAIN POLLING LOOP ===
print('=' * 60)
print('🚀 DRIVE QUEUE WORKER STARTED')
print('=' * 60)
print(f'Polling {TASKS_DIR}/ every 30 seconds...')
print(f'Results go to {RESULTS_DIR}/')
print()

# Report GPU status
status = {
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'runtime': 'active',
    'timestamp': datetime.now().isoformat()
}
with open(f'{STATUS_DIR}/status_colab.json', 'w') as f:
    json.dump(status, f, indent=2)

# Polling loop
processed = set()
iteration = 0

while True:
    iteration += 1
    
    # Find new task files
    tasks = sorted(glob.glob(f'{TASKS_DIR}/*.task.json'))
    new_tasks = [t for t in tasks if t not in processed]
    
    if new_tasks:
        print(f'\n[{datetime.now().strftime("%H:%M:%S")}] Found {len(new_tasks)} new task(s)!')
    
    for task_path in new_tasks:
        task_id = os.path.basename(task_path).replace('.task.json', '')
        print(f'\n{"="*60}')
        print(f'📋 Processing task: {task_id}')
        print(f'{"="*60}')
        
        with open(task_path) as f:
            task = json.load(f)
        
        task_type = task['type']
        params = task['params']
        print(f'  Type: {task_type}')
        print(f'  Params: {json.dumps(params, indent=4)}')
        
        # Execute
        try:
            if task_type == 'train_drl':
                result = run_drl_training(params)
            elif task_type == 'booksim_energy':
                result = run_booksim_energy(params)
            elif task_type == 'parsec_traces':
                result = run_drl_training(params)  # reuse training
            else:
                result = {'status': 'error', 'message': f'Unknown task type: {task_type}'}
            
            result['task_id'] = task_id
            result['completed'] = datetime.now().isoformat()
            
            # Write result
            result_path = f'{RESULTS_DIR}/{task_id}.result.json'
            with open(result_path, 'w') as f:
                json.dump(result, f, indent=2)
            print(f'\n✅ Result saved: {result_path}')
        
        except Exception as e:
            print(f'❌ Error: {e}')
            import traceback
            traceback.print_exc()
            result_path = f'{RESULTS_DIR}/{task_id}.error.json'
            with open(result_path, 'w') as f:
                json.dump({'task_id':task_id,'error':str(e),'status':'failed'}, f, indent=2)
        
        # Remove task file
        os.remove(task_path)
        processed.add(task_path)
    
    # Periodic status update
    if iteration % 10 == 0:
        status['last_poll'] = datetime.now().isoformat()
        with open(f'{STATUS_DIR}/status_colab.json', 'w') as f:
            json.dump(status, f, indent=2)
    
    time.sleep(30)